# Tier 2 merged analysis (four merged conditions)

Reads `derived/tier2_merged_conditions.csv` (accuracy) and `tier2_m3av_results.csv`
(timing), treating `arm_a`, `high_pass`, `hp_ffmpeg`, `hp_wiener_oa` as the four
merged conditions. Outputs tables (CSV) and figures (PNG) go to `derived/`.

In [1]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats

BASE = Path(r"C:\Users\Leon\Documents\Github\finalproject_cpe027\kaggle\results")
DERIVED = BASE / "derived"
DERIVED.mkdir(parents=True, exist_ok=True)

MERGED_PATH = DERIVED / "tier2_merged_conditions.csv"
RAW_PATH = BASE / "tier2_m3av_results.csv"

CONDITIONS = ["arm_a", "high_pass", "hp_ffmpeg", "hp_wiener_oa"]
NONBASE = ["high_pass", "hp_ffmpeg", "hp_wiener_oa"]
B = 5000
SEED = 0

In [2]:
# Report columns we expected but did not find.
merged_expected = ["tier","clip_id","duration_bucket","condition","wer","cer",
                   "audio_duration_s","rtf","seconds","nshr","kter",
                   "term_presence_recall","terms_present","language"]
raw_expected = ["tier","clip_id","duration_bucket","condition","wer","cer",
                "audio_duration_s","rtf","seconds","nshr","kter",
                "term_presence_recall","terms_present","language"]

merged_raw_cols = pd.read_csv(MERGED_PATH, nrows=0).columns.tolist()
raw_cols = pd.read_csv(RAW_PATH, nrows=0).columns.tolist()
print("Merged CSV columns:", merged_raw_cols)
print("Raw CSV columns:", raw_cols)
print("Merged missing expected:", [c for c in merged_expected if c not in merged_raw_cols])
print("Raw missing expected:", [c for c in raw_expected if c not in raw_cols])

Merged CSV columns: ['tier', 'clip_id', 'duration_bucket', 'condition', 'wer', 'cer', 'wer_min', 'wer_max', 'wer_spread', 'cer_min', 'cer_max', 'cer_spread', 'n', 'audio_duration_s']
Raw CSV columns: ['tier', 'clip_id', 'duration_bucket', 'condition', 'wer', 'cer', 'kter', 'term_presence_recall', 'terms_present', 'nshr', 'rtf', 'audio_duration_s', 'seconds', 'language']
Merged missing expected: ['rtf', 'seconds', 'nshr', 'kter', 'term_presence_recall', 'terms_present', 'language']
Raw missing expected: []


In [3]:
m = pd.read_csv(MERGED_PATH)
r = pd.read_csv(RAW_PATH)
m["wer_pct"] = 100 * m["wer"]
m["cer_pct"] = 100 * m["cer"]
print("merged rows:", len(m), "clips:", m.clip_id.nunique(), "conditions:", m.condition.nunique())
print("raw rows:", len(r), "clips:", r.clip_id.nunique(), "conditions:", r.condition.nunique())
assert set(m.condition.unique()) == set(CONDITIONS), m.condition.unique().tolist()

merged rows: 1200 clips: 300 conditions: 4
raw rows: 1800 clips: 300 conditions: 6


## 1. Accuracy table (mean WER/CER %, with n)

In [4]:
def accuracy_table(df, scope_label):
    rows = []
    for c in CONDITIONS:
        g = df[df.condition == c]
        rows.append({
            "scope": scope_label,
            "condition": c,
            "n": len(g),
            "wer_mean_pct": round(100 * g["wer"].mean(), 4),
            "cer_mean_pct": round(100 * g["cer"].mean(), 4),
        })
    return rows

acc_rows = []
acc_rows += accuracy_table(m, "overall")
for b in sorted(m.duration_bucket.unique()):
    acc_rows += accuracy_table(m[m.duration_bucket == b], b)
acc_df = pd.DataFrame(acc_rows)
print(acc_df.to_string(index=False))
acc_df.to_csv(DERIVED / "tier2_merged_accuracy.csv", index=False)

  scope    condition   n  wer_mean_pct  cer_mean_pct
overall        arm_a 300        7.5138        3.2249
overall    high_pass 300        7.5093        3.2204
overall    hp_ffmpeg 300        8.2749        3.6236
overall hp_wiener_oa 300        7.6381        3.2905
  5-10m        arm_a 160        7.4872        3.2150
  5-10m    high_pass 160        7.5008        3.1998
  5-10m    hp_ffmpeg 160        8.2667        3.6158
  5-10m hp_wiener_oa 160        7.5831        3.2440
    <5m        arm_a 140        7.5442        3.2362
    <5m    high_pass 140        7.5189        3.2440
    <5m    hp_ffmpeg 140        8.2842        3.6326
    <5m hp_wiener_oa 140        7.7010        3.3437


## 2. Paired deltas, cluster bootstrap, Wilcoxon/Holm, Cohen's dz, win/loss/tie

In [5]:
base = m[m.condition == "arm_a"].set_index("clip_id")
delta_rows = []
for c in NONBASE:
    cdf = m[m.condition == c].set_index("clip_id")
    common = base.index.intersection(cdf.index)
    for clip in common:
        delta_rows.append({
            "clip_id": clip,
            "duration_bucket": base.loc[clip, "duration_bucket"],
            "condition": c,
            "delta_wer_pp": 100 * (base.loc[clip, "wer"] - cdf.loc[clip, "wer"]),
            "delta_cer_pp": 100 * (base.loc[clip, "cer"] - cdf.loc[clip, "cer"]),
        })
delta_df = pd.DataFrame(delta_rows)
print("paired deltas:", delta_df.shape)
delta_df.to_csv(DERIVED / "tier2_merged_paired_deltas.csv", index=False)

def bootstrap_mean_ci(values, B=B, seed=SEED):
    rng = np.random.default_rng(seed)
    v = np.asarray(values, dtype=float)
    n = len(v)
    idx = rng.integers(0, n, size=(B, n))
    means = v[idx].mean(axis=1)
    return float(means.mean()), float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))

# cluster (clip-level) bootstrap over clip_id
ci_rows = []
for scope in ["overall"] + sorted(m.duration_bucket.unique()):
    dscope = delta_df if scope == "overall" else delta_df[delta_df.duration_bucket == scope]
    for c in NONBASE:
        for metric, col in [("WER", "delta_wer_pp"), ("CER", "delta_cer_pp")]:
            vals = dscope[dscope.condition == c][col].to_numpy()
            _, lo, hi = bootstrap_mean_ci(vals)
            ci_rows.append({
                "scope": scope,
                "condition": c,
                "metric": metric,
                "n_clips": len(vals),
                "mean_delta_pp": round(float(vals.mean()), 4),
                "ci_lo_pp": round(lo, 4),
                "ci_hi_pp": round(hi, 4),
            })
ci_df = pd.DataFrame(ci_rows)
print(ci_df.to_string(index=False))
ci_df.to_csv(DERIVED / "tier2_merged_bootstrap_ci.csv", index=False)

paired deltas: (900, 5)
  scope    condition metric  n_clips  mean_delta_pp  ci_lo_pp  ci_hi_pp
overall    high_pass    WER      300         0.0045   -0.0937    0.1022
overall    high_pass    CER      300         0.0045   -0.0505    0.0573
overall    hp_ffmpeg    WER      300        -0.7611   -0.9227   -0.6127
overall    hp_ffmpeg    CER      300        -0.3987   -0.4923   -0.3141
overall hp_wiener_oa    WER      300        -0.1243   -0.2259   -0.0193
overall hp_wiener_oa    CER      300        -0.0656   -0.1147   -0.0170
  5-10m    high_pass    WER      160        -0.0137   -0.1457    0.1243
  5-10m    high_pass    CER      160         0.0153   -0.0573    0.0883
  5-10m    hp_ffmpeg    WER      160        -0.7796   -1.0239   -0.5497
  5-10m    hp_ffmpeg    CER      160        -0.4007   -0.5382   -0.2741
  5-10m hp_wiener_oa    WER      160        -0.0959   -0.2258    0.0362
  5-10m hp_wiener_oa    CER      160        -0.0289   -0.0886    0.0321
    <5m    high_pass    WER      140    

In [6]:
def holm_adjust(pvals):
    p = np.asarray(pvals, dtype=float)
    m = len(p)
    order = np.argsort(p)
    adj = np.empty(m)
    for rank, i in enumerate(order):
        adj[i] = p[i] * (m - rank)
    adj_sorted = np.minimum(1.0, np.maximum.accumulate(adj[order]))
    out = np.empty(m)
    out[order] = adj_sorted
    return np.minimum(1.0, out)

sig_rows = []
for metric, col in [("WER", "delta_wer_pp"), ("CER", "delta_cer_pp")]:
    pvals = []
    for c in NONBASE:
        d = delta_df[delta_df.condition == c][col].to_numpy()
        w = stats.wilcoxon(d, alternative="two-sided", method="auto")
        dz = float(d.mean() / d.std(ddof=1)) if d.std(ddof=1) > 0 else float("nan")
        pvals.append(float(w.pvalue))
        sig_rows.append({
            "metric": metric,
            "condition": c,
            "n": len(d),
            "mean_delta_pp": round(float(d.mean()), 4),
            "cohen_dz": round(dz, 4),
            "wilcoxon_statistic": float(w.statistic),
            "p_value": w.pvalue,
            "win": int((d > 0).sum()),
            "loss": int((d < 0).sum()),
            "tie": int((d == 0).sum()),
        })
    sig_df_tmp = pd.DataFrame(sig_rows)
    # append holm per metric
# Apply Holm per metric
sig_df = pd.DataFrame(sig_rows)
for metric in ["WER", "CER"]:
    mask = sig_df.metric == metric
    sig_df.loc[mask, "p_holm"] = holm_adjust(sig_df.loc[mask, "p_value"].to_numpy())
sig_df["p_value"] = sig_df["p_value"].round(6)
sig_df["p_holm"] = sig_df["p_holm"].round(6)
print(sig_df.to_string(index=False))
sig_df.to_csv(DERIVED / "tier2_merged_significance.csv", index=False)

metric    condition   n  mean_delta_pp  cohen_dz  wilcoxon_statistic  p_value  win  loss  tie   p_holm
   WER    high_pass 300         0.0045    0.0052             18205.5 0.782489  144   128   28 0.782489
   WER    hp_ffmpeg 300        -0.7611   -0.5599              6516.0 0.000000   67   211   22 0.000000
   WER hp_wiener_oa 300        -0.1243   -0.1335             16912.0 0.033510  124   157   19 0.067020
   CER    high_pass 300         0.0045    0.0092             19159.0 0.330766  157   129   14 0.330766
   CER    hp_ffmpeg 300        -0.3987   -0.5136              8183.0 0.000000   80   218    2 0.000000
   CER hp_wiener_oa 300        -0.0656   -0.1481             18705.0 0.033080  141   154    5 0.066161


## 3. Timing table from raw CSV

In [7]:
r["merged_condition"] = r["condition"].map({
    "arm_a": "arm_a", "hp": "high_pass", "hp+sg": "high_pass",
    "hp_wiener": "high_pass", "hp_ffmpeg": "hp_ffmpeg",
    "hp_wiener_oa": "hp_wiener_oa",
})
print("Unmapped timing conditions:", r.loc[r.merged_condition.isna(), "condition"].unique().tolist())

timing_rows = []
for c in CONDITIONS:
    if c == "high_pass":
        sub = r[r.condition.isin(["hp", "hp+sg", "hp_wiener"])]
    else:
        sub = r[r.condition == c]
    timing_rows.append({
        "condition": c,
        "n": len(sub),
        "rtf_mean": round(float(sub.rtf.mean()), 4),
        "rtf_sd": round(float(sub.rtf.std(ddof=1)), 4),
        "seconds_mean": round(float(sub.seconds.mean()), 4),
        "seconds_sd": round(float(sub.seconds.std(ddof=1)), 4),
    })
timing_df = pd.DataFrame(timing_rows)
print(timing_df.to_string(index=False))

# paired overhead vs arm_a per clip (high_pass = average of hp/hp+sg/hp_wiener)
arm_t = r[r.condition == "arm_a"].set_index("clip_id")
hp_t = r[r.condition.isin(["hp", "hp+sg", "hp_wiener"])].groupby("clip_id")[["rtf", "seconds"]].mean()

overhead = {}
overhead["high_pass"] = (hp_t["rtf"] - arm_t["rtf"], hp_t["seconds"] - arm_t["seconds"])
for c in ["hp_ffmpeg", "hp_wiener_oa"]:
    c_t = r[r.condition == c].set_index("clip_id")
    overhead[c] = (c_t["rtf"] - arm_t["rtf"], c_t["seconds"] - arm_t["seconds"])

over_rows = []
for c in NONBASE:
    dr, ds = overhead[c]
    dr = dr.to_numpy(); ds = ds.to_numpy()
    mr, lorr, hirr = bootstrap_mean_ci(dr)
    ms, lors, hiss = bootstrap_mean_ci(ds)
    over_rows.append({
        "condition": c,
        "overhead_rtf_mean": round(mr, 4),
        "overhead_rtf_ci_lo": round(lorr, 4),
        "overhead_rtf_ci_hi": round(hirr, 4),
        "overhead_seconds_mean": round(ms, 4),
        "overhead_seconds_ci_lo": round(lors, 4),
        "overhead_seconds_ci_hi": round(hiss, 4),
    })
over_df = pd.DataFrame(over_rows)
timing_df = timing_df.merge(over_df, on="condition", how="left")
# baseline overhead = 0
timing_df.loc[timing_df.condition == "arm_a", ["overhead_rtf_mean","overhead_rtf_ci_lo","overhead_rtf_ci_hi",
                                              "overhead_seconds_mean","overhead_seconds_ci_lo","overhead_seconds_ci_hi"]] = 0.0
print(timing_df.to_string(index=False))
timing_df.to_csv(DERIVED / "tier2_merged_timing.csv", index=False)

Unmapped timing conditions: []
   condition   n  rtf_mean  rtf_sd  seconds_mean  seconds_sd
       arm_a 300    0.0641  0.0026       19.0719      1.2028
   high_pass 900    0.0925  0.0031       27.5076      1.5322
   hp_ffmpeg 300    0.0896  0.0028       26.6445      1.4732
hp_wiener_oa 300    0.0905  0.0029       26.9266      1.5615
   condition   n  rtf_mean  rtf_sd  seconds_mean  seconds_sd  overhead_rtf_mean  overhead_rtf_ci_lo  overhead_rtf_ci_hi  overhead_seconds_mean  overhead_seconds_ci_lo  overhead_seconds_ci_hi
       arm_a 300    0.0641  0.0026       19.0719      1.2028             0.0000              0.0000              0.0000                 0.0000                  0.0000                  0.0000
   high_pass 900    0.0925  0.0031       27.5076      1.5322             0.0284              0.0282              0.0286                 8.4359                  8.3720                  8.4983
   hp_ffmpeg 300    0.0896  0.0028       26.6445      1.4732             0.0255            

## 4. Figures (four merged conditions only)

In [8]:
fig_dir = DERIVED
m4 = m[m.condition.isin(CONDITIONS)].copy()

# mean WER by condition
means = m4.groupby("condition")["wer"].mean().reindex(CONDITIONS)
fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(means.index, 100 * means.values, color="#4c78a8")
ax.set_title("Tier 2 mean WER by merged condition")
ax.set_ylabel("WER (%)")
ax.set_xticks(range(len(means.index)))
ax.set_xticklabels(means.index, rotation=30, ha="right")
fig.tight_layout()
fig.savefig(fig_dir / "tier2_merged_wer_by_condition.png", dpi=150)
plt.close(fig)

# WER by duration bucket x condition
bucket_means = m4.groupby(["duration_bucket", "condition"])["wer"].mean().unstack("condition").reindex(columns=CONDITIONS)
fig, ax = plt.subplots(figsize=(8, 5))
bucket_means.plot.bar(ax=ax)
ax.set_title("Tier 2 WER by duration bucket and merged condition")
ax.set_ylabel("WER (%)")
fig.tight_layout()
fig.savefig(fig_dir / "tier2_merged_wer_by_bucket.png", dpi=150)
plt.close(fig)
print("Saved figures to", fig_dir)

Saved figures to C:\Users\Leon\Documents\Github\finalproject_cpe027\kaggle\results\derived


## 5. Summary

In [9]:
print("Saved tables:")
for f in ["tier2_merged_accuracy.csv","tier2_merged_paired_deltas.csv",
          "tier2_merged_bootstrap_ci.csv","tier2_merged_significance.csv",
          "tier2_merged_timing.csv"]:
    print(" -", DERIVED / f)
print("Saved figures:")
for f in ["tier2_merged_wer_by_condition.png","tier2_merged_wer_by_bucket.png"]:
    print(" -", DERIVED / f)

print()
print("Summary:")
for _, row in acc_df[acc_df.scope == "overall"].iterrows():
    print(f"  {row.condition}: WER {row.wer_mean_pct:.2f}%, CER {row.cer_mean_pct:.2f}% (n={row.n})")
print()
for _, row in ci_df[(ci_df.scope=="overall") & (ci_df.metric=="WER")].iterrows():
    print(f"  {row.condition} vs arm_a delta WER = {row.mean_delta_pp:+.3f} pp "
          f"(95% CI {row.ci_lo_pp:+.3f}, {row.ci_hi_pp:+.3f}) [n={row.n_clips}]")
print()
print("Missing expected columns:")
print("  merged:", [c for c in merged_expected if c not in merged_raw_cols])
print("  raw:", [c for c in raw_expected if c not in raw_cols])

Saved tables:
 - C:\Users\Leon\Documents\Github\finalproject_cpe027\kaggle\results\derived\tier2_merged_accuracy.csv
 - C:\Users\Leon\Documents\Github\finalproject_cpe027\kaggle\results\derived\tier2_merged_paired_deltas.csv
 - C:\Users\Leon\Documents\Github\finalproject_cpe027\kaggle\results\derived\tier2_merged_bootstrap_ci.csv
 - C:\Users\Leon\Documents\Github\finalproject_cpe027\kaggle\results\derived\tier2_merged_significance.csv
 - C:\Users\Leon\Documents\Github\finalproject_cpe027\kaggle\results\derived\tier2_merged_timing.csv
Saved figures:
 - C:\Users\Leon\Documents\Github\finalproject_cpe027\kaggle\results\derived\tier2_merged_wer_by_condition.png
 - C:\Users\Leon\Documents\Github\finalproject_cpe027\kaggle\results\derived\tier2_merged_wer_by_bucket.png

Summary:
  arm_a: WER 7.51%, CER 3.22% (n=300)
  high_pass: WER 7.51%, CER 3.22% (n=300)
  hp_ffmpeg: WER 8.27%, CER 3.62% (n=300)
  hp_wiener_oa: WER 7.64%, CER 3.29% (n=300)

  high_pass vs arm_a delta WER = +0.004 pp (95% 